In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
load_dotenv(override=True)
hf_token=os.getenv("HF_TOKEN")
api_key=os.getenv("OPENAI_API_KEY")
if api_key:
    print(f" API Key exists and begins {api_key[:8]}")
else:
    print("AI API Key not set")

 API Key exists and begins sk-proj-


In [3]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

Found 76 files in the knowledge base


In [4]:
knowledge_base=""
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        knowledge_base += f.read()
        knowledge_base += "\n\n"
print(f"total no character:{len(knowledge_base)}")

total no character:304434


In [5]:
MODEL="gpt-4.1-nano"

In [6]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 63,555


In [7]:
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)


In [8]:
len(documents)

76

In [9]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [10]:
len(chunks)

970

In [11]:
chunks[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.')

In [12]:
db_name = "./my_vector_db"

In [13]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
if os.path.exists(db_name):
    existing_db = Chroma(persist_directory=db_name, embedding_function=embeddings)
    existing_db.delete_collection()
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory=db_name
)

In [14]:
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 970 vectors with 3,072 dimensions in the vector store


In [15]:
from langchain_openai import ChatOpenAI

In [16]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0,
    model="gpt-4.1-nano"
)

In [17]:
retriever.invoke("who is avery")

[Document(id='4897ea9a-200f-42d9-a622-b248df6663fa', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000'),
 Document(id='142744c0-a115-4656-ab18-39a7abe7a84a', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial fun

In [18]:
llm.invoke("who is avery")

AIMessage(content='Avery is a given name that can be used for both males and females. It can also be a surname. The meaning of the name Avery is often associated with "ruler of the elves" or "wise." If you\'re referring to a specific person named Avery, could you please provide more context or details?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 11, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_56ee2af7a7', 'id': 'chatcmpl-EEhwiH9Z6ISe9blfXijM2ZzjYpNg6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--ae5a6b93-fdf6-4890-b11e-0a4ad5d0a4a0-0', usage_metadata={'input_tokens': 11, 'output_tokens': 63, 'total_tokens

In [19]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company insurance assistance.
You are chatting with a user about Insurance assistance.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

RAG

In [20]:
from langchain_core.messages import SystemMessage, HumanMessage


In [21]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [22]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, an insurance technology company she co-founded in 2015. She is based in San Francisco, California, and has played a key role in positioning Insurellm as a leading player in the insurance tech industry. Avery is known for her innovative leadership and expertise in risk management.'

In [23]:
import gradio as gr

In [24]:
#gr.ChatInterface(answer_question).launch()


Evaluation


In [25]:
from evaluation import test

In [26]:
tests = test.load_tests()

In [27]:
len(tests)

150

In [28]:
example=tests[0]

In [29]:
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [30]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [31]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

Chroma collection loaded: 970 documents


In [32]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.6666666666666666, ndcg=0.5561086263536886, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [33]:
eval, answer, chunks = evaluate_answer(example)

In [34]:
eval

AnswerEval(feedback="The answer correctly identifies Maxine and the award as the IIOTY (Innovator of the Year) award in 2023, matching the reference answer. It accurately states the recipient's name and the award's full name, although it abbreviates 'Innovator of the Year' as 'IIOTY' in the text, which is acceptable since it's also in the question. The answer is concise, directly addresses the question, and does not include extraneous information.", accuracy=5.0, completeness=5.0, relevance=5.0)

In [35]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The answer correctly identifies Maxine and the award as the IIOTY (Innovator of the Year) award in 2023, matching the reference answer. It accurately states the recipient's name and the award's full name, although it abbreviates 'Innovator of the Year' as 'IIOTY' in the text, which is acceptable since it's also in the question. The answer is concise, directly addresses the question, and does not include extraneous information.
5.0
5.0
5.0
